# Clustering des séries temporelles de consommation énergétique des bâtiments individuels

## Objectif du notebook

L'objectif de ce notebook est d'analyser les séries temporelles de consommation énergétique des bâtiments individuels afin d'identifier des **jours types de consommation**.

L'idée est de regrouper les journées présentant des comportements similaires en appliquant une méthode de **clustering non supervisé**. Ces profils journaliers caractéristiques permettront ensuite de mieux comprendre les habitudes de consommation des bâtiments et d'identifier différents comportements énergétiques.

---

## Approche suivie

La méthodologie est organisée en plusieurs étapes :

1. **Chargement des séries temporelles**
   - Lecture des fichiers de consommation énergétique des bâtiments individuels.
   - Sélection de la variable de consommation électrique étudiée.
   - Vérification de la fréquence temporelle des données.

2. **Transformation des séries temporelles en profils journaliers**
   - Les séries annuelles (pas de temps de 15 minutes) sont restructurées sous forme de matrices :
   Chaque ligne représente alors un profil de consommation sur une journée.

3. **Prétraitement des profils**
   - Normalisation des profils journaliers afin de comparer les formes de consommation indépendamment du niveau énergétique absolu.
   - Cette étape permet de détecter des comportements similaires même lorsque les bâtiments ont des consommations différentes.

4. **Détermination du nombre optimal de clusters**
   - Plusieurs valeurs du nombre de groupes sont testées.
   - Le score de silhouette est utilisé pour mesurer la qualité de séparation des clusters.

5. **Application du clustering**
   - Utilisation de l'algorithme **K-Means** pour regrouper les jours présentant des profils similaires.
   - Chaque cluster représente un **jour type de consommation**.

6. **Analyse et visualisation des résultats**
   - Visualisation des centroïdes des clusters correspondant aux journées représentatives.
   - Analyse de la fréquence d'apparition de chaque type de journée.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from kneed import KneeLocator
import seaborn as sns

ROOT = Path().resolve().parent.parent
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

COL = "out.electricity.total.energy_consumption..kwh"
steps_per_day = 96

WEATHER_COLS = [
    "out.outdoor_air_drybulb_temp..c",
    "out.outdoor_air_relative_humidity..percentage",
    "out.outdoor_air_wetbulb_temp..c",
    "out.outdoor_humidity_ratio..kgwater_per_kgdryair",
]

## Exploration rapide d'un bâtiment (sanity check visuel)

In [ ]:
df = pd.read_parquet(DATA_RAW / "347201-0.parquet")
df.head()

In [ ]:
dff = (
    pd.read_parquet(DATA_RAW / "347201-0.parquet")
    .assign(timestamp=lambda x: x["timestamp"] - pd.Timedelta("15m"))
    .set_index("timestamp")
    .loc[:, lambda x: x.columns.str.match(r"out\.electricity\..*\.energy_consumption\.\.kwh")]
    .loc[:, lambda x: x.ne(0).any(axis=0)]
    .rename(columns=lambda x: x[16:-24])
    .drop(columns="net")
)

dfh = dff.resample("h").sum()
dfd = dfh.resample("D").sum()

fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
dfd.drop(columns="total").plot(ax=ax, kind="area", cmap="tab20")
plt.legend(loc="upper right")
plt.show()

In [ ]:
piv = dfh.pivot_table(index=dfh.index.normalize(), columns=dfh.index.hour, values="total")
cmap = (piv.index.day_of_week // 5).map({0: "tab:blue", 1: "tab:red"})
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
piv.T.plot(ax=ax, color=cmap, alpha=0.1, legend=False)
plt.show()

## Fonction d'extraction jour × features (un bâtiment)

In [ ]:
def build_day_features(parquet_path, col=COL, weather_cols=WEATHER_COLS):
    """Extrait les jours (forme + amplitude + météo) pour un bâtiment."""
    d = pd.read_parquet(parquet_path)
    ts = d[col].values
    n_days_ = len(ts) // steps_per_day
    ts = ts[: n_days_ * steps_per_day]

    dates_ = pd.to_datetime(d["timestamp"]) if "timestamp" in d.columns else pd.to_datetime(d.index)
    dates_ = dates_[: n_days_ * steps_per_day]

    days_ = ts.reshape(n_days_, steps_per_day)
    dates_daily = (
        pd.Series(dates_)
        .groupby(pd.Series(np.arange(len(dates_))) // steps_per_day)
        .first().dt.normalize().values
    )

    mask_valid_ = ~np.isnan(days_).any(axis=1)
    days_ = days_[mask_valid_]
    dates_daily = dates_daily[mask_valid_]

    # Forme
    mean_ = days_.mean(axis=1, keepdims=True)
    std_ = days_.std(axis=1, keepdims=True)
    std_[std_ == 0] = 1e-8
    shape_ = (days_ - mean_) / std_

    # Amplitude
    amplitude_ = days_.sum(axis=1)

    meta = {
        "bldg_id": Path(parquet_path).stem,
        "date": dates_daily,
        "amplitude": amplitude_,
    }

    for wcol in weather_cols:
        prefix = wcol.replace("out.", "").replace("..", "_").replace(".", "_")
        if wcol in d.columns:
            weather = d[wcol].values[: n_days_ * steps_per_day].reshape(n_days_, steps_per_day)[mask_valid_]
            meta[f"{prefix}_mean"] = weather.mean(axis=1)
            meta[f"{prefix}_min"] = weather.min(axis=1)
            meta[f"{prefix}_max"] = weather.max(axis=1)
        else:
            meta[f"{prefix}_mean"] = np.nan
            meta[f"{prefix}_min"] = np.nan
            meta[f"{prefix}_max"] = np.nan

    return pd.DataFrame(meta), shape_, days_, dates_daily

## Extraction multi-bâtiments

In [ ]:
parquet_files = sorted(glob.glob(str(DATA_RAW / "*-0.parquet")))

all_meta, all_shapes, all_days_raw = [], [], []

for f in parquet_files:
    meta_f, shape_f, days_raw_f, _ = build_day_features(f)
    all_meta.append(meta_f)
    all_shapes.append(shape_f)
    all_days_raw.append(days_raw_f)

meta_all = pd.concat(all_meta, ignore_index=True)
shape_all = np.vstack(all_shapes)
days_raw_all = np.vstack(all_days_raw)

print("meta :", meta_all.shape)
print("shape:", shape_all.shape)
print("Nombre de bâtiments :", meta_all["bldg_id"].nunique())

## Clustering exploratoire mono-bâtiment (forme vs amplitude) sur le bâtiment de référence

In [ ]:
bldg_ref = "347201-0"
mask_ref = meta_all["bldg_id"].values == bldg_ref

days_clean = days_raw_all[mask_ref]
day_dates_clean = pd.to_datetime(meta_all.loc[mask_ref, "date"].values)
shape_ref = shape_all[mask_ref]

day_mean = days_clean.mean(axis=1, keepdims=True)
day_std = days_clean.std(axis=1, keepdims=True)
day_std[day_std == 0] = 1e-8

X = (days_clean - day_mean) / day_std          # forme
Y = StandardScaler().fit_transform(days_clean)  # amplitude/niveau global

print(X.shape, Y.shape)

## Fonctions utilitaires (coude/silhouette, PCA, projection)

In [ ]:
def plot_cluster_profiles(X, title1, title2):
    k_range = range(2, 15)
    inertias, silhouettes = [], []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        sil = silhouette_score(X, labels, sample_size=5000, random_state=42)
        silhouettes.append(sil)
        print(f"k={k:2d} | inertia={km.inertia_:10.1f} | silhouette={sil:.4f}")

    k_opt = KneeLocator(list(k_range), inertias, curve="convex", direction="decreasing").elbow
    print(f"K optimal (coude) : {k_opt}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(list(k_range), inertias, marker='o')
    axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertie"); axes[0].set_title(title1)
    axes[1].plot(list(k_range), silhouettes, marker='o', color='orange')
    axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].set_title(title2)
    plt.tight_layout()
    plt.show()
    return k_opt


def cluster_days(X_input, day_dates_input, k):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(X_input)

    results = pd.DataFrame({"date": day_dates_input, "cluster": cluster_labels})
    results["weekday"] = results["date"].dt.day_name()
    results["is_weekend"] = results["date"].dt.dayofweek >= 5
    results["month"] = results["date"].dt.month

    print(results["cluster"].value_counts().sort_index())
    return kmeans, cluster_labels, results


def compute_pca(X_input, n_components=10, titre1=""):
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_input)
    var_ratio = pca.explained_variance_ratio_
    var_cum = np.cumsum(var_ratio)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(range(1, len(var_ratio) + 1), var_ratio)
    axes[0].set_xlabel("Composante"); axes[0].set_ylabel("Variance expliquée")
    axes[0].set_title("Variance expliquée par composante")
    axes[1].plot(range(1, len(var_cum) + 1), var_cum, marker="o")
    axes[1].axhline(0.90, color="red", linestyle="--", label="90 %")
    axes[1].set_xlabel("Nb composantes"); axes[1].set_ylabel("Variance cumulée")
    axes[1].legend(); axes[1].set_title(f"Variance cumulée {titre1}")
    plt.tight_layout()
    plt.show()

    n_components_90 = np.argmax(var_cum >= 0.90) + 1
    print(f"Composantes pour 90% variance {titre1}: {n_components_90}")
    return X_pca, pca, var_ratio, var_cum, n_components_90


def plot_pca_clusters(X_pca, cluster_labels, var_ratio, title="Projection PCA"):
    fig, ax = plt.subplots(figsize=(8, 7))
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="tab10", s=8, alpha=0.6)
    ax.set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}%)")
    ax.set_title(title)
    plt.colorbar(scatter, ax=ax, label="Cluster")
    plt.tight_layout()
    plt.show()

## Choix de k et clustering (forme vs amplitude, bâtiment de référence)

In [ ]:
k_forme  = plot_cluster_profiles(X, "Coude - forme", "Silhouette - forme")
k_amplet = plot_cluster_profiles(Y, "Coude - amplitude", "Silhouette - amplitude")

kmeans_forme, cluster_labels_forme, results_forme = cluster_days(X, day_dates_clean, k_forme)
kmeans_amplet, cluster_labels_amplet, results_amplet = cluster_days(Y, day_dates_clean, k_amplet)

## Visualisation interactive des profils moyens

In [ ]:
def plot_interactive_cluster_profiles(cluster_labels, k_final, titre):
    fig = go.Figure()
    hours = np.linspace(0, 24, steps_per_day)

    for c in range(k_final):
        mask = cluster_labels == c
        cluster_profiles = days_clean[mask]
        mean_profile = cluster_profiles.mean(axis=0)
        std_profile = cluster_profiles.std(axis=0)

        fig.add_trace(go.Scatter(
            x=hours, y=mean_profile, mode="lines",
            name=f"Cluster {c} (n={mask.sum()})", visible=(c == 0), line=dict(width=3)
        ))
        fig.add_trace(go.Scatter(
            x=np.concatenate([hours, hours[::-1]]),
            y=np.concatenate([mean_profile + std_profile, (mean_profile - std_profile)[::-1]]),
            fill="toself", mode="lines", opacity=0.15, name=f"Std {c}",
            visible=(c == 0), showlegend=False, hoverinfo="skip"
        ))

    buttons = []
    for c in range(k_final):
        visibility = []
        for i in range(k_final):
            visibility += [i == c, i == c]
        buttons.append(dict(label=f"Cluster {c}", method="update",
                             args=[{"visible": visibility}, {"title": f"{titre} - Cluster {c}"}]))
    buttons.append(dict(label="Tous", method="update",
                         args=[{"visible": [True] * (2 * k_final)}, {"title": f"{titre} (k={k_final})"}]))

    fig.update_layout(
        title=titre, xaxis_title="Heure", yaxis_title="Consommation (kWh)",
        template="plotly_white", width=1000, height=600,
        updatemenus=[dict(buttons=buttons, direction="down", x=1.05, y=1, showactive=True)]
    )
    fig.show()

plot_interactive_cluster_profiles(cluster_labels_amplet, k_amplet, "Profil par amplitude")
plot_interactive_cluster_profiles(cluster_labels_forme, k_forme, "Profil par forme")

## Composition calendaire des clusters

In [ ]:
def plot_cluster_calendar_composition(results, titre):
    comp_weekend = pd.crosstab(results["cluster"], results["is_weekend"], normalize="index") * 100
    comp_weekend.columns = ["Semaine (%)", "Weekend (%)"]
    comp_month = pd.crosstab(results["cluster"], results["month"], normalize="index") * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    comp_weekend.plot(kind="bar", stacked=True, ax=axes[0], colormap="Set2")
    axes[0].set_title("Semaine / weekend par cluster")
    sns.heatmap(comp_month, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1])
    axes[1].set_title("Répartition mensuelle par cluster (%)")
    fig.suptitle(titre, fontsize=14)
    plt.tight_layout()
    plt.show()
    return comp_weekend, comp_month

plot_cluster_calendar_composition(results_amplet, "Composition (amplitude)")
plot_cluster_calendar_composition(results_forme, "Composition (forme)")

## Frise temporelle

In [ ]:
def plot_cluster_timeline(results, k_final, titre):
    results_sorted = results.sort_values("date").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(16, 3))
    scatter = ax.scatter(results_sorted["date"], [1] * len(results_sorted),
                          c=results_sorted["cluster"], cmap="tab10", s=15)
    ax.set_yticks([])
    ax.set_title(titre)
    plt.colorbar(scatter, ax=ax, label="Cluster", ticks=range(k_final))
    plt.tight_layout()
    plt.show()

plot_cluster_timeline(results_forme, k_forme, "Timeline clusters (forme)")
plot_cluster_timeline(results_amplet, k_amplet, "Timeline clusters (amplitude)")

## PCA sur forme et amplitude (bâtiment de référence)

In [ ]:
X_pca_forme, pca_forme, var_ratio_forme, var_cum_forme, _ = compute_pca(X, n_components=10, titre1="(Forme)")
Y_pca_amplet, pca_amplet, var_ratio_amplet, var_cum_amplet, _ = compute_pca(Y, n_components=10, titre1="(Amplet)")

plot_pca_clusters(X_pca_forme, cluster_labels_forme, var_ratio_forme, "PCA clusters (forme)")
plot_pca_clusters(Y_pca_amplet, cluster_labels_amplet, var_ratio_amplet, "PCA clusters (amplitude)")

## Dataset multivarié multi-bâtiments (forme + amplitude z-scorée/bâtiment + météo globale)

In [ ]:
weather_feature_cols = [c for c in meta_all.columns if c.endswith(("_mean", "_min", "_max"))]
print("Colonnes météo :", weather_feature_cols)

mask_ok = meta_all[weather_feature_cols].notna().all(axis=1).values
print(f"Jours conservés : {mask_ok.sum()} / {len(mask_ok)}")

shape_ok = shape_all[mask_ok]
meta_ok = meta_all[mask_ok].reset_index(drop=True)

print("Nombre de bâtiments :", meta_ok["bldg_id"].nunique())

In [ ]:
# Amplitude : log + z-score PAR BÂTIMENT (isole le jour type de la taille du bâtiment)
meta_ok["amp_log"] = np.log1p(meta_ok["amplitude"])

amp_stats = meta_ok.groupby("bldg_id")["amp_log"].agg(amp_log_mean="mean", amp_log_std="std")
amp_stats["amp_log_std"] = amp_stats["amp_log_std"].replace(0, np.nan).fillna(1e-8)

meta_ok = meta_ok.merge(amp_stats, on="bldg_id", how="left")
meta_ok["amplitude_z"] = (meta_ok["amp_log"] - meta_ok["amp_log_mean"]) / meta_ok["amp_log_std"]

# Météo : standardisation globale
weather_scaler = StandardScaler()
weather_scaled = weather_scaler.fit_transform(meta_ok[weather_feature_cols].values)

scalar_features = np.column_stack([meta_ok["amplitude_z"].values, weather_scaled])
X_multi = np.hstack([shape_ok, scalar_features])

print(X_multi.shape)

## PCA + choix de k (multivarié multi-bâtiments)

In [ ]:
X_multi_pca, pca_multi, var_ratio_multi, var_cum_multi, _ = compute_pca(
    X_multi, n_components=15, titre1="(Multivarié multi-bâtiments)"
)

k_multi_opt = plot_cluster_profiles(
    X_multi_pca,
    "Coude - Clustering multivarié multi-bâtiments",
    "Silhouette - Clustering multivarié multi-bâtiments"
)

## Clustering final multivarié

In [ ]:
kmeans_multi = KMeans(n_clusters=k_multi_opt, random_state=42, n_init=20)
cluster_labels_multi = kmeans_multi.fit_predict(X_multi_pca)

meta_ok["cluster_multi"] = cluster_labels_multi
meta_ok["weekday"] = pd.to_datetime(meta_ok["date"]).dt.day_name()
meta_ok["is_weekend"] = pd.to_datetime(meta_ok["date"]).dt.dayofweek >= 5
meta_ok["month"] = pd.to_datetime(meta_ok["date"]).dt.month

print(meta_ok["cluster_multi"].value_counts().sort_index())

plot_pca_clusters(X_multi_pca, cluster_labels_multi, var_ratio_multi,
                   title=f"Clustering multivarié multi-bâtiments, k={k_multi_opt}")

## Vérification : un cluster n'est pas dominé par un seul bâtiment

In [ ]:
comp_bldg = pd.crosstab(meta_ok["cluster_multi"], meta_ok["bldg_id"], normalize="index") * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(comp_bldg, cmap="YlGnBu", ax=ax)
ax.set_title("Répartition (%) des bâtiments par cluster")
plt.tight_layout()
plt.show()

## Analyse des clusters multivariés

In [ ]:
main_weather_col = next((c for c in weather_feature_cols if c.endswith("_mean")), weather_feature_cols[0])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=meta_ok, x="cluster_multi", y=main_weather_col, ax=axes[0])
axes[0].set_title(f"{main_weather_col} par cluster")
sns.boxplot(data=meta_ok, x="cluster_multi", y="amplitude_z", ax=axes[1])
axes[1].set_title("Amplitude relative (z-score/bâtiment) par cluster")
comp_weekend_multi = pd.crosstab(meta_ok["cluster_multi"], meta_ok["is_weekend"], normalize="index") * 100
comp_weekend_multi.columns = ["Semaine (%)", "Weekend (%)"]
comp_weekend_multi.plot(kind="bar", stacked=True, ax=axes[2], colormap="Set2")
axes[2].set_title("Semaine / weekend par cluster")
plt.tight_layout()
plt.show()

In [ ]:
mean_cols = [c for c in weather_feature_cols if c.endswith("_mean")] + ["amplitude_z"]
cluster_summary = meta_ok.groupby("cluster_multi")[mean_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cluster_summary.T, annot=True, fmt=".1f", cmap="coolwarm", ax=ax)
ax.set_title("Moyenne des variables par cluster")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
hours = np.linspace(0, 24, steps_per_day)
colors = plt.cm.tab10(np.linspace(0, 1, k_multi_opt))
for c in range(k_multi_opt):
    mask = cluster_labels_multi == c
    ax.plot(hours, shape_ok[mask].mean(axis=0), label=f"Cluster {c} (n={mask.sum()})", color=colors[c], linewidth=2)
ax.set_xlabel("Heure"); ax.set_ylabel("Consommation normalisée (forme)")
ax.set_title("Profils de forme moyens par cluster (multi-bâtiments)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Sauvegarde de tous les artefacts

In [ ]:
"""meta_ok.to_parquet(DATA_PROCESSED / "clustering_multivarie_jours_types.parquet", index=False)

joblib.dump(kmeans_multi, DATA_PROCESSED / "kmeans_multivarie.joblib")
joblib.dump(pca_multi, DATA_PROCESSED / "pca_multivarie.joblib")
joblib.dump(weather_scaler, DATA_PROCESSED / "weather_scaler.joblib")
joblib.dump(amp_stats, DATA_PROCESSED / "amp_stats.joblib")
joblib.dump(weather_feature_cols, DATA_PROCESSED / "weather_feature_cols.joblib")
"""

## Reverse : classifier un nouveau jour dans un cluster existant

In [ ]:
def classify_new_day(
    profile_kwh,
    weather_values,
    bldg_id=None,
    amp_stats=amp_stats,
    weather_scaler=weather_scaler,
    weather_feature_cols=weather_feature_cols,
    pca=pca_multi,
    kmeans=kmeans_multi,
):
    """
    profile_kwh : array (96,) — profil brut 15min du jour à classer
    weather_values : dict {nom_colonne_meteo: valeur}
    bldg_id : identifiant du bâtiment
    """

    profile_kwh = np.asarray(profile_kwh, dtype=np.float64)

    assert profile_kwh.shape[0] == steps_per_day, \
        f"Le profil doit avoir {steps_per_day} points"


    # Forme
    m = profile_kwh.mean()
    s = profile_kwh.std()

    if s == 0:
        s = 1e-8

    shape_new = (profile_kwh - m) / s


    # Amplitude
    amp_log_new = np.log1p(profile_kwh.sum())

    if bldg_id is not None and bldg_id in amp_stats.index:
        mu = amp_stats.loc[bldg_id, "amp_log_mean"]
        sd = amp_stats.loc[bldg_id, "amp_log_std"]
    else:
        mu = amp_stats["amp_log_mean"].mean()
        sd = amp_stats["amp_log_std"].mean()

    if sd == 0:
        sd = 1e-8

    amp_z = (amp_log_new - mu) / sd


    # Météo
    weather_vec = np.array(
        [[weather_values[c] for c in weather_feature_cols]],
        dtype=np.float64
    )

    weather_scaled_new = weather_scaler.transform(weather_vec)[0]


    # Assemblage
    scalar_vec = np.concatenate(
        [[amp_z], weather_scaled_new]
    )

    full_vec = np.concatenate(
        [shape_new, scalar_vec]
    ).reshape(1, -1)

    full_vec = full_vec.astype(np.float64)


    # PCA + KMeans
    vec_pca = pca.transform(full_vec).astype(np.float64)

    cluster = kmeans.predict(vec_pca)[0]

    return int(cluster)



# Test sur un jour
idx_test = 0

row_test = meta_ok.iloc[idx_test]

bldg_test = row_test["bldg_id"]

weather_test = {
    c: row_test[c]
    for c in weather_feature_cols
}

mask_bldg_full = meta_all["bldg_id"].values == bldg_test

profile_test = days_raw_all[mask_bldg_full][0]


predicted = classify_new_day(
    profile_test,
    weather_test,
    bldg_id=bldg_test
)

print(
    f"Cluster prédit : {predicted} | "
    f"Cluster réel : {row_test['cluster_multi']}"
)



def classify_new_days_batch(
    df_days,
    profile_col="profile",
    weather_cols=weather_feature_cols,
    bldg_col="bldg_id"
):
    """
    df_days :
    - colonne profile contenant des array(96,)
    - colonnes météo
    - colonne bldg_id optionnelle
    """

    predictions = []

    for _, row in df_days.iterrows():

        weather_vals = {
            c: row[c]
            for c in weather_cols
        }

        bldg = row[bldg_col] if bldg_col in df_days.columns else None

        pred = classify_new_day(
            row[profile_col],
            weather_vals,
            bldg_id=bldg
        )

        predictions.append(pred)


    df_out = df_days.copy()

    df_out["cluster_predicted"] = predictions

    return df_out

## Test de cohérence (reclasser un jour déjà connu)

In [ ]:
idx_test = 0
row_test = meta_ok.iloc[idx_test]
bldg_test = row_test["bldg_id"]
weather_test = {c: row_test[c] for c in weather_feature_cols}

# Récupération du profil brut correspondant (même ligne dans days_raw_all, avant filtrage météo)
mask_bldg_full = meta_all["bldg_id"].values == bldg_test
profile_test = days_raw_all[mask_bldg_full][0]  # à ajuster pour pointer exactement la même date que row_test

predicted = classify_new_day(profile_test, weather_test, bldg_id=bldg_test)
print(f"Cluster prédit : {predicted} | Cluster réel : {row_test['cluster_multi']}")

In [ ]:
def classify_new_days_batch(df_days, profile_col="profile", weather_cols=weather_feature_cols, bldg_col="bldg_id"):
    """
    df_days : DataFrame avec une colonne `profile_col` contenant des array(96,),
              les colonnes météo, et éventuellement `bldg_col`.
    Retourne le DataFrame avec une colonne 'cluster_predicted' ajoutée.
    """
    predictions = []
    for _, row in df_days.iterrows():
        weather_vals = {c: row[c] for c in weather_cols}
        bldg = row[bldg_col] if bldg_col in df_days.columns else None
        pred = classify_new_day(row[profile_col], weather_vals, bldg_id=bldg)
        predictions.append(pred)

    df_out = df_days.copy()
    df_out["cluster_predicted"] = predictions
    return df_out